In [1]:
!pip install -q tensorflow

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

In [3]:
# load the pretrained model
model = VGG16(weights='imagenet', include_top=False, pooling = 'avg')

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
im1_path = "/content/Gemini_Generated_Image_sletu2sletu2slet (1).png"
im2_path = "/content/Gemini_Generated_Image_sletu2sletu2slet (2).png"

In [5]:
# resize the images for VGG16 input to 224*224
img_1 = Image.open(im1_path).convert("RGB").resize((224,224))
img_2 = Image.open(im2_path).convert("RGB").resize((224,224))

In [6]:
def compute_pixel_distance(img_1, img_2):
  arr1 = np.array(img_1, dtype = np.float32)
  arr2 = np.array(img_2, dtype = np.float32)
  return np.mean((arr1-arr2)**2)

mse_score = compute_pixel_distance(img_1, img_2)
print("MSE Score: ", mse_score)

MSE Score:  6466.0415


In [7]:
mse_score = compute_pixel_distance(img_1, img_1)
print("MSE Score: ", mse_score)

MSE Score:  0.0


In [8]:
mse_score = compute_pixel_distance(img_2, img_2)
print("MSE Score: ", mse_score)

MSE Score:  0.0


In [10]:
from platform import processor
def get_image_embeddings(img):
  # PIL image is getting converted to numpy array
  x = image.img_to_array(img)
  # I am expanding one dimension to make the input dimension to this model as (1,224,224,3)
  # since the model accepts the input as a batch of images, the input numpy array should
  # also be of a batch shape (b, m,n, c)
  x = np.expand_dims(x, axis=0)
  # preprocessing of the image as per VGG 16 input
  x = preprocess_input(x)
  embeddings = model.predict(x, verbose=0)
  return embeddings

emb1 = get_image_embeddings(img_1)
emb2 = get_image_embeddings(img_2)


In [12]:
cosine = cosine_similarity(emb1, emb2)
print("Cosine Similarity: ", cosine)

Cosine Similarity:  [[0.9355994]]


## Vision Transformers

In [13]:
import torch
import numpy as np
from transformers import AutoImageProcessor, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

In [14]:
model_ckpt = "google/vit-base-patch16-224"
# vision transformer from google where 16*16 patches were created on an input image 224*224
processor = AutoImageProcessor.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)
model.eval() # using the model for predictions

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

In [15]:
im1_path = "/content/Gemini_Generated_Image_sletu2sletu2slet (1).png"
im2_path = "/content/Gemini_Generated_Image_sletu2sletu2slet (2).png"

In [16]:
def get_image_embedding(img):
    # Preprocess the image (resizing, normalization) to match model expectations
    inputs = processor(images=img, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the [CLS] token embedding which represents the entire image.
    # It is located at the 0th index of the sequence length dimension.
    embedding = outputs.last_hidden_state[:, 0, :].numpy()
    return embedding

In [18]:
img_a = Image.open(im1_path).convert("RGB").resize((224, 224))
img_b = Image.open(im2_path).convert("RGB").resize((224, 224))
emb_a = get_image_embedding(img_a)
emb_b = get_image_embedding(img_b)

In [20]:
cosine = cosine_similarity(emb_a, emb_b)
print("Cosine Similarity: ", cosine)

Cosine Similarity:  [[0.869969]]
